In [ ]:
from pyspark.sql.functions import col, current_timestamp

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_sellers_table_name = dbutils.widgets.get("raw_olist_sellers_table")

silver_schema = dbutils.widgets.get("silver_schema")
sellers_table_name = dbutils.widgets.get("sellers_table")

In [ ]:
raw_olist_sellers_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_sellers_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{sellers_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{sellers_table_name} (
            sellerId STRING,
            sellerZipCodePrefix STRING,
            sellerCity STRING,
            sellerState STRING,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
sellers_silver_df = (
    raw_olist_sellers_df.where(col("seller_id").rlike("^[0-9a-fA-F]{32}$"))
    .select(
        col("seller_id").cast("string").alias("sellerId"),
        col("seller_zip_code_prefix").cast("string").alias("sellerZipCodePrefix"),
        col("seller_city").cast("string").alias("sellerCity"),
        col("seller_state").cast("string").alias("sellerState"),
    )
    .withColumn("processedTimestamp", current_timestamp())
    .dropDuplicates(["sellerId"])
)

In [ ]:
sellers_silver_df.createOrReplaceTempView("sellers_silver_view")

spark.sql(f"""
    MERGE INTO {catalog}.{silver_schema}.{sellers_table_name} AS target
    USING sellers_silver_view AS source
    ON target.sellerId = source.sellerId
    WHEN MATCHED THEN
        UPDATE SET 
            target.sellerZipCodePrefix = source.sellerZipCodePrefix,
            target.sellerCity = source.sellerCity,
            target.sellerState = source.sellerState,
            target.processedTimestamp = source.processedTimestamp
    WHEN NOT MATCHED THEN
        INSERT *
    """)